# Von Wörtern zu Embeddings – Konzepte

**Business Problem:** E-Mail-Texte in Zahlen umwandeln und anschliessend entscheiden, welche Nachrichten auf Spam überprüft werden müssen.

Später werden dieselben Schritte verwendet, um Stellenanzeigen in Jobkategorien einzuordnen.

Wichtig: Ein Modell unterstützt die Entscheidungsfindung, trifft aber nicht selbst die geschäftliche Entscheidung.

## Lernziele

- Häufigkeiten, gewichtete Häufigkeiten und gelernte Wortvektoren unterscheiden.
- Ein Skalarprodukt und die Kosinus-Ähnlichkeit berechnen.
- Erklären, welche Informationen bei der statischen Wortzuordnung und der Mittelwertbildung verloren gehen.

## Datensatz und praktischer Umfang

Die kurzen Nachrichten und numerischen Wortvektoren unten sind konstruierte Beispiele. Die Spam-Demonstration verwendet erfundene Vektoren, wie auf Folie 11, und trainiert einen echten Klassifikator mit synthetischen Labels. Sie trainiert weder word2vec noch reproduziert sie Ergebnisse mit echten E-Mails.

Der gesamte Code wird bereitgestellt. Führe die Zellen von oben nach unten aus.

## Bibliotheken und Einstellungen

Verwende den vorbereiteten Python 3.11 Codespace-Kernel. Es wird weder eine GPU noch ein API-Schlüssel benötigt.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Eine lexikalische Baseline

TF-IDF gewichtet Wortidentitäten. Unterschiedliche Schreibweisen bleiben unterschiedliche Koordinaten. Das Vokabular und die Wortgewichte werden ausschliesslich aus dem Trainingstext gelernt.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
texts = ['claim prize', 'claim reward', 'meeting team']
vectorizer = TfidfVectorizer()
counts = vectorizer.fit_transform(texts)
display(pd.DataFrame(counts.toarray(), columns=vectorizer.get_feature_names_out()))
display(pd.DataFrame(cosine_similarity(counts), index=texts, columns=texts).round(3))

,claim,meeting,prize,reward,team
0,0.605349,0.000000,0.795961,0.000000,0.000000
1,0.605349,0.000000,0.000000,0.795961,0.000000
2,0.000000,0.707107,0.000000,0.000000,0.707107


,claim prize,claim reward,meeting team
claim prize,1.000,0.366,0.0
claim reward,0.366,1.000,0.0
meeting team,0.000,0.000,1.0


## 2. Wie word2vec lernt

Skip-gram verwendet ein zentrales Wort, um benachbarte Wörter vorherzusagen. Eine gelernte Input-Tabelle liefert einen Wortvektor; eine separate Kontext-Tabelle liefert Vorhersage-Scores. Softmax wandelt die Scores in Wahrscheinlichkeiten für Kontextwörter um. Die Loss-Funktion verwendet ein beobachtetes benachbartes Wort, kein Spam- oder Jobkategorie-Label.

In [3]:
scores = np.array([2.0, 0.0, 1.0, -1.0, -2.0])
probabilities = np.exp(scores - scores.max())
probabilities /= probabilities.sum()
display(pd.DataFrame({'word': ['claim', 'prize', 'reward', 'meeting', 'team'],
                      'constructed score': scores, 'context probability': probabilities}))

,word,constructed score,context probability
0,claim,2.0,0.636409
1,prize,0.0,0.086129
2,reward,1.0,0.234122
3,meeting,-1.0,0.031685
4,team,-2.0,0.011656


## 3. Bedeutung als Distanz: Kosinus-Ähnlichkeit vorsichtig verwenden

Ein Skalarprodukt multipliziert die entsprechenden Einträge und addiert sie. Die Kosinus-Ähnlichkeit teilt das Ergebnis durch die beiden Vektorlängen. Sie ist weder eine Wahrscheinlichkeit noch ein Beweis für Synonymie oder eine kausale Erklärung.

In [4]:
a, b = np.array([1.0, 2.0]), np.array([0.0, 1.0])
dot_product = a @ b
cosine = dot_product / (np.linalg.norm(a) * np.linalg.norm(b))
print('Dot product:', dot_product, '| Cosine:', round(cosine, 3))

Dot product: 2.0 | Cosine: 0.894


## 4. Wortvektoren zu einem Dokumentvektor mitteln

Die folgende Nachschlagetabelle ist erfunden. Der Mittelwert derselben statischen Wortvektoren ignoriert die Reihenfolge.

In [5]:
word_vectors = {'bank': [1., 0.], 'paid': [0., 1.], 'supplier': [1., 1.]}
examples = ['bank paid supplier', 'supplier paid bank']
document_vectors = np.array([
    np.mean([word_vectors[word] for word in text.split()], axis=0) for text in examples])
display(pd.DataFrame(document_vectors, index=examples, columns=['value 1', 'value 2']))

,value 1,value 2
bank paid supplier,0.666667,0.666667
supplier paid bank,0.666667,0.666667


## Interpretation

Wer bezahlt wen in jedem Beispiel? Warum kann der statische Mittelwert diesen Unterschied nicht erkennen? Ein kontextueller Encoder kann die Positionen vor dem Pooling berücksichtigen, garantiert aber keine Korrektheit.

## 5. Zurück zu den Folien: Welche E-Mails müssen auf Spam überprüft werden?

**Durchgerechnetes Beispiel:** E-Mail-Text → mittlerer Wortvektor → logistische Regression → geschätzte Spam-Wahrscheinlichkeit → Schwellenwert → Überprüfung.

Auf Folie 11 gilt `claim = [0, 1]` und `prize = [1, 2]`, sodass `claim prize` zu `[0.5, 1.5]` wird. Wir behalten diese exakten Werte unten bei. Die anderen Vektoren und alle Nachrichten-Labels sind für Lehrzwecke erfunden und keine gelernten word2vec-Ausgaben. Ein echtes word2vec-Modell würde seine Nachschlagetabelle aus Trainingstexten lernen, ohne Spam-Labels; der Klassifikator würde anschliessend aus gelabelten Trainings-E-Mails lernen.

Diese kurzen Phrasen stehen stellvertretend für vollständige E-Mails. Trainings-, Validierungs- und Testphrasen sind getrennt, aber bewusst einfach gehalten. Es werden weder echte E-Mail-Daten noch ein Modelldownload benötigt.

In [6]:
spam_word_vectors = {
    'claim': [0., 1.], 'prize': [1., 2.], 'reward': [1., 1.8],
    'win': [0.8, 1.8], 'free': [0.5, 1.5], 'offer': [0.5, 1.2],
    'meeting': [-1., -1.], 'team': [-1., -0.5],
    'project': [-0.8, -0.7], 'report': [-0.6, -1.]
}

def spam_email_mean(text):
    return np.mean([spam_word_vectors[word] for word in text.split()], axis=0)

spam_parts = {
    'train': pd.DataFrame([
        ('claim prize', 1), ('win reward', 1), ('free prize', 1),
        ('claim offer', 1), ('win free prize', 1), ('reward offer', 1),
        ('team meeting', 0), ('project report', 0), ('team project', 0),
        ('meeting report', 0), ('team meeting report', 0), ('project meeting', 0)
    ], columns=['text', 'spam']),
    'validation': pd.DataFrame([
        ('claim reward', 1), ('free offer', 1), ('win prize', 1),
        ('team report', 0), ('project team meeting', 0), ('report meeting project', 0)
    ], columns=['text', 'spam']),
    'test': pd.DataFrame([
        ('claim free reward', 1), ('win offer', 1), ('free reward', 1),
        ('meeting team project', 0), ('report project team', 0), ('team reward', 0)
    ], columns=['text', 'spam'])
}
spam_vectors = {name: np.array([spam_email_mean(text) for text in frame['text']])
                for name, frame in spam_parts.items()}
print('Slide 11: claim prize →', spam_email_mean('claim prize'))
display(pd.DataFrame({'messages': {name: len(frame) for name, frame in spam_parts.items()}}))

Slide 11: claim prize → [0.5 1.5]


,messages
train,12
validation,6
test,6


### Den Klassifikator trainieren, nicht die Worttabelle

Beginne mit einer Baseline, die immer «kein Spam» vorhersagt: Es wird nichts überprüft. Vergleiche TF-IDF plus logistische Regression mit logistischer Regression auf den E-Mail-Mittelwerten mit jeweils zwei Werten. TF-IDF lernt sein Vokabular und seine Gewichtungen ausschliesslich aus den Trainingsnachrichten. Die erfundene Worttabelle bleibt unverändert.

Eine Spam-Wahrscheinlichkeit stammt vom trainierten Klassifikator, nicht von der Kosinus-Ähnlichkeit. Bei diesem kleinen, künstlich erstellten Datensatz sollte sie als Modellschätzung betrachtet werden und nicht als zuverlässiges Risiko in der realen Welt.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score

spam_tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(random_state=SEED))
spam_tfidf.fit(spam_parts['train']['text'], spam_parts['train']['spam'])
spam_classifier = LogisticRegression(random_state=SEED)
spam_classifier.fit(spam_vectors['train'], spam_parts['train']['spam'])
spam_probabilities = {
    name: {
        'Always normal': np.zeros(len(spam_parts[name])),
        'TF-IDF': spam_tfidf.predict_proba(spam_parts[name]['text'])[:, 1],
        'Illustrative vector mean': spam_classifier.predict_proba(spam_vectors[name])[:, 1]
    }
    for name in ['validation', 'test']
}

### Einen Schwellenwert für die Überprüfung anhand der Validierungsnachrichten wählen

Angenommen, eine unnötige Überprüfung einer normalen Nachricht kostet **5 Einheiten**, während das Übersehen von Spam **1 Einheit** kostet. Dies sind erfundene Kosten und keine Empfehlung für ein echtes E-Mail-System. Teste drei Schwellenwerte anhand der Validierungsnachrichten und wähle für jedes Modell den Schwellenwert mit den niedrigsten Validierungskosten; bei Gleichstand wird der erste Schwellenwert beibehalten. Werte das Modell anschliessend einmal anhand der Testnachrichten aus.

**Precision** ist der Anteil der markierten Nachrichten, die tatsächlich Spam sind. **Recall** ist der Anteil aller Spam-Nachrichten, die wir markieren. Keine der beiden Kennzahlen allein beschreibt den Überprüfungsaufwand oder dessen Kosten.

In [8]:
def spam_review_cost(labels, predictions):
    normal_ok, false_alarms, missed_spam, spam_found = confusion_matrix(
        labels, predictions, labels=[0, 1]).ravel()
    return 5 * false_alarms + missed_spam

thresholds = [0.3, 0.5, 0.7]
spam_rows = []
selected_thresholds = {}
for name, probabilities in spam_probabilities['validation'].items():
    costs = [spam_review_cost(spam_parts['validation']['spam'], probabilities >= t)
             for t in thresholds]
    threshold = thresholds[int(np.argmin(costs))]
    selected_thresholds[name] = threshold
    predicted = spam_probabilities['test'][name] >= threshold
    actual = spam_parts['test']['spam']
    spam_rows.append({
        'model': name, 'threshold': threshold, 'validation cost': min(costs),
        'test precision': precision_score(actual, predicted, zero_division=0),
        'test recall': recall_score(actual, predicted, zero_division=0),
        'test reviews': int(predicted.sum()), 'test cost': spam_review_cost(actual, predicted)
    })
display(pd.DataFrame(spam_rows).set_index('model').round(3))
spam_test = spam_parts['test'].copy()
spam_test['estimated spam probability'] = spam_probabilities['test']['Illustrative vector mean']
spam_test['send to review'] = spam_test['estimated spam probability'] >= selected_thresholds['Illustrative vector mean']
display(spam_test.round(3))

,threshold,validation cost,test precision,test recall,test reviews,test cost
model,,,,,,
Always normal,0.3,3,0.00,0.0,0,3
TF-IDF,0.5,0,0.75,1.0,4,5
Illustrative vector mean,0.3,0,0.75,1.0,4,5


,text,spam,estimated spam probability,send to review
0,claim free reward,1,0.879,True
1,win offer,1,0.901,True
2,free reward,1,0.924,True
3,meeting team project,0,0.104,False
4,report project team,0,0.115,False
5,team reward,0,0.624,True


### Diskussion: Eine Belohnung kann auch legitim sein

- Die Testnachricht `team reward` ist als normal gelabelt. Markiert der Klassifikator mit gemittelten Wortvektoren sie? Untersuche die entsprechende Zeile, anstatt anzunehmen, dass eine dichte Repräsentation besser abschneidet.
- Die Baseline, die immer «normal» vorhersagt, erzeugt keine Fehlalarme, übersieht aber jede Spam-Nachricht. Hier wird ihre Precision als null angezeigt, weil sie keine Nachrichten markiert.
- Ein niedrigerer Schwellenwert führt normalerweise dazu, dass mehr Nachrichten überprüft werden müssen. Würdest du dieselben Kosten beibehalten, wenn das Übersehen von Spam einen erheblichen Verlust verursachen könnte?

**Interpretation dieses Durchlaufs:** Beide trainierten Modelle erkennen alle drei Spam-Nachrichten, markieren aber auch `team reward`. Dieser Fehlalarm kostet 5 Einheiten und damit mehr als die 3 Einheiten für die übersehenen Spam-Nachrichten bei der Baseline, die immer «normal» vorhersagt. Ein höherer Recall bedeutet nicht automatisch ein besseres Geschäftsergebnis. Passe den Schwellenwert nicht anhand dieser Testergebnisse erneut an; eine überarbeitete Strategie benötigt neue Evaluationsdaten.

Die Worttabelle wurde so gestaltet, dass sie die meisten dieser Phrasen voneinander unterscheiden kann. Dieser Vergleich kann weder belegen, dass word2vec besser als TF-IDF ist, noch dass seine Wahrscheinlichkeiten zuverlässig sind. Echte E-Mails enthalten neue Wörter, sich verändernde Sprache und mehrdeutige Kontexte; eine Phrase ausserhalb dieser kleinen Nachschlagetabelle benötigt eine andere Verarbeitungsstrategie. Die Toy-Funktion akzeptiert bewusst nur die aufgelisteten Wörter.

**Nächster praktischer Schritt:** Die [Übung zur Jobkategorisierung](text_embeddings_exercise.ipynb) verwendet denselben Ablauf Repräsentation → Klassifikator → Evaluation, lernt word2vec aber tatsächlich aus Trainingstexten. Das Trainieren und Evaluieren eines echten Spam-Detektors ist eine optionale Erweiterung, die einen autorisierten, gelabelten Textkorpus erfordert; die E-Mail-CSV-Datei mit ausschliesslich Merkmalen enthält nicht die dafür benötigten Wörter.

## Zusammenfassung und geschäftliche Auswirkungen

Behalte TF-IDF als Baseline bei. Word2vec lernt verwandte Verwendungsmuster, keine garantierte Bedeutung. Trainiere einen separaten Klassifikator und teste ihn anhand neuer Dokumente.

## Einschränkungen

Dies ist ein kleines Lehrbeispiel und kein Nachweis für die Leistung im praktischen Einsatz. Halte die Evaluation vom Training und der Modellauswahl getrennt. Überprüfe die Fehler und die Entscheidung des Nutzers, nicht nur eine einzelne Kennzahl. Die Spam-Nachschlagetabelle und die Kosten sind erfunden, und sechs Testnachrichten reichen nicht aus, um eine Entscheidung über den praktischen Einsatz zu treffen.

## Weiterführende Übungen

- Trainiere word2vec und wähle deine eigenen Wörter zur Visualisierung in der [Wortkarten-Übung](word_relationships_exercise.ipynb), inspiriert von Seite 17 aus *Slides_MSC_Data_Science_Week_08_EN.pdf*. Vergleiche die Darstellung mit den Kosinus-Ähnlichkeiten, die anhand aller 100 Werte pro Wort berechnet wurden.
- Erkläre, warum gemeinsame Kontexte auch Antonyme miteinander verbinden können.
- Was sollte passieren, wenn eine Nachricht keine bekannten Wörter enthält?
- Bearbeite die Übung zur Jobkategorisierung; ihr kleiner synthetischer Datensatz ist kein Massstab für den geschäftlichen Einsatz.

## Referenzen

**Pflichtlektüre:** Stanford CS224N, Frühling 2024, Vorlesung zu Wortvektoren, bereitgestellter Auszug.
**Referenz der Lehrperson:** Mikolov et al. (2013), *Effiziente Schätzung von Wortrepräsentationen im Vektorraum*.
https://arxiv.org/abs/1301.3781
**Offizielle Dokumentation:** https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction

**### Jupyter Notebook - Abschlussinformationen**

Gib bei der Abgabe deinen Namen, das Abgabedatum und eine kurze Beschreibung deiner Änderungen an.